# Match Cells
Use the saved global propensity model from `global_psm.ipynb` to predict a propensity score for each of the candidate treatment and control cells, and match each treatment cell to a set of control cells with similar propensity scores.

In [ ]:
# Select PA
site_id = 352159

In [ ]:
from pathlib import Path
import sys
import os
import ee
import geemap
import numpy as np
import pandas as pd
import geopandas as gpd

cur = Path.cwd().resolve()
for parent in [cur] + list(cur.parents):
    if parent.name == "tpae":
        os.chdir(parent)
        break

sys.path.insert(0, str((Path.cwd() / "src").resolve()))

from utils.variables import (
    PROJECT,
    EE_CRS_METERS,
    PSM_CELL_SIZE,
    BIOME_ASSET_ID,
    HGFC_ASSET_ID,
    COVARIATES,
)

from absolute_effectiveness.site_selector import SiteSelector
from psm.prepare_pa_grid import load_pa_candidate_cells
from psm.covariates import build_resampled_covariates
from psm.cell_features import extract_cells_with_covariates
from psm.match_cells import (
    add_propensity_scores,
    match_treatment_control,
    filter_matched_grids,
    save_matching_outputs,
)

ee.Authenticate()
ee.Initialize(project=PROJECT)

site_selector = SiteSelector()

EE_CRS_1km = ee.Projection(EE_CRS_METERS).atScale(PSM_CELL_SIZE)

## Data Prep
Load candidate cells and covariates.

In [ ]:
pa_ctx = load_pa_candidate_cells(site_id, site_selector)
PA_ID = pa_ctx["PA_ID"]
test_sites = pa_ctx["test_sites"]
site_geom = pa_ctx["site_geom"]
treatment_cells = pa_ctx["treatment_cells"]
control_cells = pa_ctx["control_cells"]
grid_fc = pa_ctx["grid_fc"]
covariates = build_resampled_covariates(EE_CRS_1km)

## Predict Propensity Scores
Calculate covariate values within cells and apply propensity model to predict propensity scores for each cell.

In [ ]:
grid_fc, cells_df = extract_cells_with_covariates(grid_fc, covariates, EE_CRS_1km)
cells_df = add_propensity_scores(cells_df)

## Match Cells
Match each treatment cell to a set of control cells. Re-use of control cells is ok. Matching is based on similarity of propensity score, with exact-matching required for country and ecoregion. Ecoregion constraint relaxes to biome if no within-ecoregion controls exist.

In [ ]:
match_df, treat_df, control_df = match_treatment_control(cells_df)
matched_grids = filter_matched_grids(grid_fc, match_df)
save_matching_outputs(matched_grids, match_df, PA_ID)

## Diagnostics

In [ ]:
# === Diagnostic 1: Stratum coverage ===
# Checks whether each treatment stratum has enough controls available for matching.

print(f"PA {PA_ID} — Stratum Coverage")
print("=" * 60)

treat_df = cells_df[cells_df["protected"] == 1]
control_df = cells_df[cells_df["protected"] == 0]

stratum_summary = []
for (country, ecoregion), treat_sub in treat_df.groupby(["country", "ecoregion"]):
    biome = treat_sub["biome"].iloc[0]
    
    n_same_eco = len(control_df[(control_df["country"] == country) & (control_df["ecoregion"] == ecoregion)])
    n_same_biome = len(control_df[(control_df["country"] == country) & (control_df["biome"] == biome)])
    n_same_country = len(control_df[control_df["country"] == country])
    
    stratum_summary.append({
        "country": country,
        "ecoregion": int(ecoregion),
        "biome": int(biome),
        "n_treatment": len(treat_sub),
        "n_same_ecoregion": n_same_eco,
        "n_same_biome": n_same_biome,
        "n_same_country": n_same_country,
        "coverage": "ecoregion" if n_same_eco >= 4 else ("biome" if n_same_biome >= 4 else "INSUFFICIENT"),
    })

stratum_df = pd.DataFrame(stratum_summary)
print(stratum_df.to_string(index=False))

# Flags
insufficient = stratum_df[stratum_df["coverage"] == "INSUFFICIENT"]
if len(insufficient) > 0:
    n_unmatchable = insufficient["n_treatment"].sum()
    print(f"\n⚠ {len(insufficient)} stratum/strata have insufficient controls — {n_unmatchable} treatment cells likely unmatchable")

needs_fallback = stratum_df[stratum_df["coverage"] == "biome"]
if len(needs_fallback) > 0:
    n_fallback = needs_fallback["n_treatment"].sum()
    print(f"⚠ {len(needs_fallback)} stratum/strata will fall back to biome — {n_fallback} treatment cells affected")

In [ ]:
# === Diagnostic 2: Propensity score & matching quality ===
# Inspects the propensity score distribution and the resulting matches.

print(f"PA {PA_ID} — Propensity Scores & Matching")
print("=" * 60)

t_scores = cells_df.loc[cells_df["protected"] == 1, "propensity_score"]
c_scores = cells_df.loc[cells_df["protected"] == 0, "propensity_score"]

print(f"\nPropensity score distribution:")
print(f"                    treatment    control     diff")
print(f"  mean:              {t_scores.mean():>8.4f}   {c_scores.mean():>8.4f}   {t_scores.mean() - c_scores.mean():>+8.4f}")
print(f"  min:               {t_scores.min():>8.4f}   {c_scores.min():>8.4f}")
print(f"  max:               {t_scores.max():>8.4f}   {c_scores.max():>8.4f}")

# Distribution sanity checks
if t_scores.mean() <= c_scores.mean():
    print(f"\n⚠ Treatment mean propensity ≤ control mean — model may be inverted for this PA")

t_range = t_scores.max() - t_scores.min()
overlap_low = max(t_scores.min(), c_scores.min())
overlap_high = min(t_scores.max(), c_scores.max())
overlap_pct = max(0, (overlap_high - overlap_low) / t_range) if t_range > 0 else 0
print(f"\nOverlap region (where matching can find candidates): [{overlap_low:.3f}, {overlap_high:.3f}]")
print(f"Treatment cells in overlap region: {((t_scores >= overlap_low) & (t_scores <= overlap_high)).sum()}/{len(t_scores)} ({((t_scores >= overlap_low) & (t_scores <= overlap_high)).mean():.1%})")

# Matching outcomes
print(f"\nMatching results:")
n_unique_treat_matched = match_df["treat_cell_id"].nunique()
n_unique_control_used = match_df["control_cell_id"].nunique()
n_treatment_total = len(treat_df)
n_control_total = len(control_df)

print(f"  Total matched pairs:           {len(match_df)}")
print(f"  Treatment match coverage:      {n_unique_treat_matched}/{n_treatment_total} ({n_unique_treat_matched/n_treatment_total:.1%})")
print(f"  Unique controls used:          {n_unique_control_used}/{n_control_total} ({n_unique_control_used/n_control_total:.1%})")
print(f"  Mean matches per treatment:    {len(match_df) / n_unique_treat_matched:.2f}")

control_reuse = match_df.groupby("control_cell_id").size()
print(f"  Control reuse:                 mean={control_reuse.mean():.1f}, max={control_reuse.max()}, median={control_reuse.median():.0f}")

print(f"\n  Propensity distance stats:")
print(f"    mean:  {match_df['ps_distance'].mean():.4f}")
print(f"    max:   {match_df['ps_distance'].max():.4f}")
print(f"    >0.05: {(match_df['ps_distance'] > 0.05).sum()} pairs ({(match_df['ps_distance'] > 0.05).mean():.1%})")

print(f"\n  Fallback usage:")
fallback_counts = match_df["match_fallback"].fillna("ecoregion").value_counts()
for label, n in fallback_counts.items():
    print(f"    {label}: {n} pairs ({n/len(match_df):.1%})")

# Red flags
if n_unique_treat_matched / n_treatment_total < 0.8:
    print(f"\n⚠ Match coverage <80% — significant fraction of PA unanalyzable")
if control_reuse.mean() > 10:
    print(f"⚠ Mean control reuse >10 — control pool may be too small for reliable matching")
if (match_df["match_fallback"] == "biome").mean() > 0.5:
    print(f"⚠ >50% of matches used biome fallback — ecoregion constraint may be too tight")

In [ ]:
# === Diagnostic 3: Training data extrapolation ===
# Checks whether PA covariates fall within the range of the training data.
# Predictions outside training range are extrapolation and should be treated with caution.

print(f"PA {PA_ID} — Covariate Extrapolation Check")
print("=" * 60)

training_data = pd.read_parquet("data/samples_thinned.parquet")

if training_data is not None:
    print(f"\n{'covariate':<20} {'PA range':<25} {'Training p1-p99':<25} {'Out of range':<15}")
    print("-" * 85)
    
    any_extrapolation = False
    for col in ["elevation", "slope", "treecover2000", "travel_time", "log_pop_density"]:
        pa_vals = cells_df[col]
        train_vals = training_data[col]
        
        # Use 1st-99th percentile of training data as the "valid" range
        # (more robust than absolute min/max which is sensitive to outliers)
        train_p1 = train_vals.quantile(0.01)
        train_p99 = train_vals.quantile(0.99)
        
        n_low = (pa_vals < train_p1).sum()
        n_high = (pa_vals > train_p99).sum()
        n_out = n_low + n_high
        
        pa_range_str = f"[{pa_vals.min():.1f}, {pa_vals.max():.1f}]"
        train_range_str = f"[{train_p1:.1f}, {train_p99:.1f}]"
        out_str = f"{n_out} ({n_out/len(pa_vals):.0%})"
        
        if n_out > 0:
            any_extrapolation = True
            out_str += " ⚠"
        
        print(f"{col:<20} {pa_range_str:<25} {train_range_str:<25} {out_str:<15}")
    
    if any_extrapolation:
        print(f"\n⚠ Some PA cells have covariate values outside the training distribution.")
        print(f"  Propensity scores for these cells are extrapolations and may be unreliable.")

In [ ]:
# === Diagnostic 4: Covariate balance check ===
# Use the standardized differences test (Feng et al. 2022) to check that covariates
# are balanced between treatment and control cells after matching.
# This test verifies the validity of the PSM.

# Build matched treatment and control DataFrames
# Each row of match_df becomes one row in each: treat row has treatment covariates,
# control row has control covariates. Controls appear multiple times (reuse).
matched_treat = match_df.merge(
    cells_df[["cell_ID"] + COVARIATES],
    left_on="treat_cell_id",
    right_on="cell_ID",
).drop(columns="cell_ID")

matched_control = match_df.merge(
    cells_df[["cell_ID"] + COVARIATES],
    left_on="control_cell_id",
    right_on="cell_ID",
).drop(columns="cell_ID")

# REPLACE the compute_smd function with:
def compute_pair_smd(matched_t_vals, matched_c_vals, full_t_vals, full_c_vals):
    """
    Pair-level standardized mean difference.

    Returns the mean absolute pair distance (in pooled-SD units) plus
    the 90th percentile, which surfaces worst-pair imbalance.

    Pooled SD is computed on the FULL (unmatched) treatment and control pools
    to anchor the metric to the original covariate scale.

    Parameters
    ----------
    matched_t_vals, matched_c_vals : pandas Series, equal length
        Covariate values for matched treatment and control cells (in pair order).
    full_t_vals, full_c_vals : pandas Series
        Covariate values for the full unmatched treatment and control pools.
        Used to compute the pooled SD that anchors the metric.

    Returns
    -------
    (mean_abs_smd, p90_abs_smd, signed_smd) : tuple of floats
    """
    var_full_t = full_t_vals.var()
    var_full_c = full_c_vals.var()
    pooled_sd = np.sqrt((var_full_t + var_full_c) / 2)
    if pooled_sd == 0:
        return 0.0, 0.0, 0.0

    pair_diffs = (matched_t_vals.values - matched_c_vals.values) / pooled_sd
    return (
        np.abs(pair_diffs).mean(),
        np.percentile(np.abs(pair_diffs), 90),
        pair_diffs.mean(),  # signed mean for directional info
    )

print("Pair-level balance check")
print("Mean absolute pair SMD (lower = better individual match quality)")
print("Threshold: < 0.25 = acceptable; < 0.10 = excellent")
print("=" * 90)
print(f"{'covariate':<20s} {'mean |smd|':>12s} {'p90 |smd|':>12s} {'signed mean':>14s} {'verdict':>20s}")
print("-" * 90)

unmatched_treat = cells_df[cells_df["protected"] == 1]
unmatched_control = cells_df[cells_df["protected"] == 0]

for col in COVARIATES:
    mean_abs, p90, signed = compute_pair_smd(
        matched_treat[col],
        matched_control[col],
        unmatched_treat[col],
        unmatched_control[col],
    )

    if mean_abs < 0.10:
        verdict = "excellent"
    elif mean_abs < 0.25:
        verdict = "acceptable"
    elif mean_abs < 0.5:
        verdict = "IMBALANCED"
    else:
        verdict = "IMBALANCED"

    print(f"{col:<20s} {mean_abs:>12.3f} {p90:>12.3f} {signed:>+14.3f} {verdict:>20s}")

In [ ]:
# === Diagnostic 6: Covariate extrapolation check ===
# Do covariate values of treatment cells fall outside the range of covariate values of control cells?
# If so, covariates cannot be balanced by any matching method.
print(f"\nExtrapolation check (PA {PA_ID}):")
covariates_list = ["elevation", "slope", "treecover2000", "travel_time", "log_pop_density"]
for col in covariates_list:
    c_min = control_df[col].min()
    c_max = control_df[col].max()
    n_below = (treat_df[col] < c_min).sum()
    n_above = (treat_df[col] > c_max).sum()
    n_total = n_below + n_above
    pct = n_total / len(treat_df)
    flag = " ⚠" if pct > 0.10 else ""
    print(f"  {col:20s}: control range [{c_min:>8.2f}, {c_max:>8.2f}], "
          f"{n_total:>4} treatment cells out of range ({pct:>5.1%}){flag}")

## Visualization

In [ ]:
Map = geemap.Map()
Map.add_basemap("CartoDB.Positron")
ecoRegions = ee.FeatureCollection(BIOME_ASSET_ID)

color_updates = [
    {"ECO_ID": 204, "COLOR": '#B3493B'},
    {"ECO_ID": 245, "COLOR": '#267400'},
    {"ECO_ID": 259, "COLOR": '#004600'},
    {"ECO_ID": 286, "COLOR": '#82F178'},
    {"ECO_ID": 316, "COLOR": '#E600AA'},
    {"ECO_ID": 453, "COLOR": '#5AA500'},
    {"ECO_ID": 317, "COLOR": '#FDA87F'},
    {"ECO_ID": 763, "COLOR": '#A93800'},
]

def add_style_property(feature):
    color = feature.get('COLOR')
    return feature.set('style', {'color': color, 'width': 0})
ecoRegions = ecoRegions.map(add_style_property)

for update in color_updates:
    layer = ecoRegions.filter(ee.Filter.eq('ECO_ID', update['ECO_ID'])).map(
        lambda f: f.set({'COLOR': update['COLOR'], 'style': {'color': update['COLOR'], 'width': 0}})
    )
    ecoRegions = ecoRegions.filter(ee.Filter.neq('ECO_ID', update['ECO_ID'])).merge(layer)

ecoRegions = ecoRegions.style(**{'styleProperty': 'style'})

land_mask = (
    ee.Image(HGFC_ASSET_ID)
    .select("datamask")
    .eq(1)  # 1 = land, 2 = permanent water/ocean, 0 = no data
)

Map.addLayer(ecoRegions.updateMask(land_mask), {}, 'Ecoregions')
Map.addLayer(grid_fc, {"color": "black"}, "Candidate Cells")
Map.addLayer(matched_grids, {"color": "yellow"}, "Matched Cells")
Map.centerObject(grid_fc)
Map
